# Context Memory Architecture

This demo combines a sliding window, a running summary, and vector retrieval to remember details from many turns back.

In [ ]:
from pathlib import Path
import os
import sys

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / 'llm_lab').exists():
        sys.path.insert(0, str(candidate))
        repo_root = candidate
        break

from llm_lab.memory import MemoryManager

print(f'Repo root: {repo_root}')

In [ ]:
def summarize_turns(turns):
    facts = []
    for turn in turns[-10:]:
        if any(keyword in turn.content.lower() for keyword in ['favorite', 'name', 'city', 'project', 'deadline']):
            facts.append(f'{turn.role}: {turn.content}')
    return ' ; '.join(facts) or 'No durable facts extracted yet.'

manager = MemoryManager(window_size=6, summarizer=summarize_turns)
for index in range(20):
    if index == 3:
        manager.add_turn('user', 'My favorite city is Bengaluru and I will mention it again later.')
    elif index == 15:
        manager.add_turn('user', 'I also need a reminder that the favorite city was Bengaluru.')
    else:
        manager.add_turn('user', f'Turn {index}: keep this note in memory.')
    manager.add_turn('assistant', f'Confirmed turn {index}.')

print(manager.build_context('What city did I say I liked?', top_k=3))

In [ ]:
def reply(message, history):
    manager.add_turn('user', message)
    context = manager.build_context(message, top_k=3)
    hits = manager.retrieve(message, top_k=1)
    answer = hits[0].text if hits else 'I do not have a matching memory yet.'
    manager.add_turn('assistant', answer)
    return answer + '

Context used:
' + context

reply('What was my favorite city?', [])

Hook `reply` into `gradio.ChatInterface` to turn the same memory manager into a simple chatbot UI.